In [0]:
#Import libraries
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, IntegerType
from pyspark.sql.functions import trim,col

In [0]:
# Read data bronze table and load to dataframe
df = spark.table("baraa_projectwork.bronze.crm_prd_info")

display(df)

In [0]:
# Remove extra spaces from all string columns
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

In [0]:
# Extracts the first 5 characters from 'prd_key', replaces '-' with '_', and assigns to 'cat_id'
df = df.withColumn("cat_id", F.regexp_replace(F.substring(col("prd_key"), 1, 5), "-", "_"))

# Updates 'prd_key' to contain characters from position 7 to the end
df = df.withColumn("prd_key", F.substring(col("prd_key"), 7, F.length(col("prd_key"))))


In [0]:
# Replace null values in 'prd_cost' column with 0
df = df.withColumn("prd_cost", F.coalesce(col("prd_cost"), F.lit(0)))
display(df)

In [0]:
df = (
    df.withColumn(
        "prd_line",
        F.when(F.upper(F.col("prd_line")) == "M", "Mountain")
         .when(F.upper(F.col("prd_line")) == "R", "Road")
         .when(F.upper(F.col("prd_line")) == "S", "Other Products")
         .when(F.upper(F.col("prd_line")) == "T", "Touring")
         .otherwise("n/a")
    )
)
display(df)


In [0]:
df = df.withColumn("prd_start_dt", col("prd_start_dt").cast(DateType()))

In [0]:
RENAME_MAP = {
    "prd_id": "product_id",
    "cat_id": "category_id",
    "prd_key": "product_number",
    "prd_nm": "product_name",
    "prd_cost": "product_cost",
    "prd_line": "product_line",
    "prd_start_dt": "start_date",
    "prd_end_dt": "end_date"
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

### Writing to Silver table


In [0]:
df.limit(10).display()

In [0]:

df.write.mode("overwrite").format("delta").saveAsTable("baraa_projectwork.silver.crm_products")

In [0]:
%sql
SELECT * FROM baraa_projectwork.silver.crm_products LIMIT 10